# LightGBM Market-Making Regime Model — Tree Pipeline

Parallel to the LSTM pipeline. Trains two LightGBM regressors (volatility, OFI)
in a **two-stage workflow**:

| Stage | Data | Features | Purpose |
|-------|------|----------|---------|
| 1 | Subsampled (every 4th row) | All ~103 | Feature importance → prune to top 90% |
| 2 | Full resolution | Pruned set | Production models saved to disk |

**Output files** (written to `tree/`):
- `vol_model_final.txt` — LightGBM volatility model
- `ofi_model_final.txt` — LightGBM OFI model  
- `selected_features.json` — feature names used by both models

In [1]:
pip install lightgbm numpy pandas torch scikit-learn tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Cell 0: Configuration & Imports ─────────────────────────────────────────

import glob
import json
import os
import time
from pathlib import Path
from datetime import datetime

import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_PATH   = "/workspace/data/*.pt"       # <-- update if needed
OUTPUT_DIR  = Path(".")                    # writes relative to notebook dir

VOL_MODEL_PATH  = OUTPUT_DIR / "vol_model_final.txt"
OFI_MODEL_PATH  = OUTPUT_DIR / "ofi_model_final.txt"
FEATURES_PATH   = OUTPUT_DIR / "selected_features.json"

# ── Feature-engineering constants ────────────────────────────────────────────
COLUMN_NAMES    = ["price", "volume", "spread", "OFI", "direction", "volatility", "price_change"]
LAG_RANGE       = list(range(1, 31))
ROLLING_WINDOWS = [5, 10, 15, 20, 30, 50]
EWMA_SPANS      = [5, 10, 15, 20, 30, 50]
WARMUP_ROWS     = 50
SUBSAMPLE_STEP  = 4
CUMULATIVE_THRESHOLD = 0.90

# ── LightGBM hyper-parameters ────────────────────────────────────────────────
LGB_PARAMS = dict(
    n_estimators=10000,        # ← was 3000 (ran as 6000, hit cap — now uncapped)
    learning_rate=0.03,
    num_leaves=31,
    max_depth=7,
    min_child_samples=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
EARLY_STOPPING_ROUNDS = 75
LOG_PERIOD            = 100

def _ts():
    """Return current timestamp string for log lines."""
    return datetime.now().strftime("[%H:%M:%S]")

print(f"{_ts()} ✅ Configuration loaded")
print(f"   DATA_PATH   : {DATA_PATH}")
print(f"   OUTPUT_DIR  : {OUTPUT_DIR.resolve()}")
print(f"   LGB params  : n_estimators={LGB_PARAMS['n_estimators']}, "
      f"lr={LGB_PARAMS['learning_rate']}, num_leaves={LGB_PARAMS['num_leaves']}")

# ── Cross-validation settings (for best-iteration finding in Stage 2) ────────
N_CV_FOLDS = 5

# Stage 1 uses fewer estimators — we only need reliable feature importances,
# not perfectly tuned trees. Keeps the exploratory pass fast.
LGB_STAGE1_PARAMS = {**LGB_PARAMS, "n_estimators": 500}

# Low-level LightGBM params for lgb.cv() (maps sklearn names -> native names)
LGB_CV_PARAMS = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    LGB_PARAMS["learning_rate"],
    "num_leaves":       LGB_PARAMS["num_leaves"],
    "max_depth":        LGB_PARAMS["max_depth"],
    "min_data_in_leaf": LGB_PARAMS["min_child_samples"],
    "bagging_fraction": LGB_PARAMS["subsample"],
    "bagging_freq":     1,
    "feature_fraction": LGB_PARAMS["colsample_bytree"],
    "lambda_l1":        LGB_PARAMS["reg_alpha"],
    "lambda_l2":        LGB_PARAMS["reg_lambda"],
    "seed":             LGB_PARAMS["random_state"],
    "num_threads":      -1,
    "verbose":          -1,
}

print(f"{_ts()} CV folds: {N_CV_FOLDS} | Stage1 estimators: {LGB_STAGE1_PARAMS['n_estimators']}")


[20:15:43] ✅ Configuration loaded
   DATA_PATH   : /workspace/data/*.pt
   OUTPUT_DIR  : /
   LGB params  : n_estimators=10000, lr=0.03, num_leaves=31
[20:15:43] CV folds: 5 | Stage1 estimators: 500


## Cell 1 — Data Loading

In [3]:
# ── Cell 1: Data Loading ────────────────────────────────────────────────────

def load_all_files(data_path: str) -> list:
    """Return sorted list of all .pt files found at data_path."""
    all_files = sorted(glob.glob(data_path))
    if not all_files:
        raise FileNotFoundError(f"No .pt files found at {data_path}")
    print(f"{_ts()} Total files: {len(all_files)}")
    return all_files


def load_and_concat(file_list: list) -> np.ndarray:
    """Load .pt tensor files and concatenate into a single float32 array."""
    arrays = []
    t0 = time.time()
    for i, fpath in enumerate(file_list):
        tensor = torch.load(fpath, map_location="cpu", weights_only=True)
        arrays.append(tensor.numpy().astype(np.float32))
        elapsed = time.time() - t0
        print(f"  {_ts()} [{i+1:02d}/{len(file_list)}] {os.path.basename(fpath)}"
              f"  shape={arrays[-1].shape}  ({elapsed:.1f}s elapsed)")

    result = np.concatenate(arrays, axis=0)
    print(f"{_ts()} Concatenated - shape: {result.shape} | "
          f"{result.nbytes / 1e9:.2f} GB | {time.time()-t0:.1f}s total")
    return result


# ── Run data loading ──────────────────────────────────────────────────────────
print(f"\n{_ts()} === DATA LOADING ===")
all_files = load_all_files(DATA_PATH)

print(f"\n{_ts()} Loading ALL data...")
t_load = time.time()
train_data = load_and_concat(all_files)
print(f"{_ts()} Data load done in {time.time()-t_load:.1f}s")


[20:15:43] === DATA LOADING ===
[20:15:43] Total files: 230

[20:15:43] Loading ALL data...
  [20:15:43] [01/230] tensor_OFI_Enhanced_BTC_2023-05-16.pt  shape=(527472, 7)  (0.0s elapsed)
  [20:15:43] [02/230] tensor_OFI_Enhanced_BTC_2023-05-17.pt  shape=(947035, 7)  (0.0s elapsed)
  [20:15:43] [03/230] tensor_OFI_Enhanced_BTC_2023-05-18.pt  shape=(1380768, 7)  (0.1s elapsed)
  [20:15:43] [04/230] tensor_OFI_Enhanced_BTC_2023-05-19.pt  shape=(909060, 7)  (0.1s elapsed)
  [20:15:43] [05/230] tensor_OFI_Enhanced_BTC_2023-05-20.pt  shape=(461962, 7)  (0.1s elapsed)
  [20:15:43] [06/230] tensor_OFI_Enhanced_BTC_2023-05-21.pt  shape=(635300, 7)  (0.1s elapsed)
  [20:15:43] [07/230] tensor_OFI_Enhanced_BTC_2023-05-22.pt  shape=(839344, 7)  (0.1s elapsed)
  [20:15:43] [08/230] tensor_OFI_Enhanced_BTC_2023-05-23.pt  shape=(941171, 7)  (0.2s elapsed)
  [20:15:43] [09/230] tensor_OFI_Enhanced_BTC_2023-05-24.pt  shape=(1291594, 7)  (0.2s elapsed)
  [20:15:43] [10/230] tensor_OFI_Enhanced_BTC_2023

## Cell 2 — Feature Engineering

In [4]:
# ── Cell 2: Feature Engineering ─────────────────────────────────────────────

def build_features(data: np.ndarray, selected_features: list = None) -> pd.DataFrame:
    """
    Build the full ~103-column feature set from raw (N, 7) data.
    If selected_features is provided, only compute those columns (Stage 2).
    All features use only past values — no lookahead.
    """
    df = pd.DataFrame(data, columns=COLUMN_NAMES).astype(np.float32)

    need_all = selected_features is None
    keep = set(selected_features) if selected_features else set()

    def _need(col_name):
        return need_all or col_name in keep

    for lag in LAG_RANGE:
        col = f"OFI_lag_{lag}"
        if _need(col):
            df[col] = df["OFI"].shift(lag).astype(np.float32)

    for lag in LAG_RANGE:
        col = f"vol_lag_{lag}"
        if _need(col):
            df[col] = df["volatility"].shift(lag).astype(np.float32)

    for w in ROLLING_WINDOWS:
        if _need(f"OFI_rolling_mean_{w}"):
            df[f"OFI_rolling_mean_{w}"] = df["OFI"].rolling(w, min_periods=w).mean().astype(np.float32)
        if _need(f"OFI_rolling_std_{w}"):
            df[f"OFI_rolling_std_{w}"]  = df["OFI"].rolling(w, min_periods=w).std().astype(np.float32)

    for s in EWMA_SPANS:
        if _need(f"OFI_ewma_{s}"):
            df[f"OFI_ewma_{s}"] = df["OFI"].ewm(span=s, adjust=False).mean().astype(np.float32)

    for w in ROLLING_WINDOWS:
        if _need(f"vol_rolling_mean_{w}"):
            df[f"vol_rolling_mean_{w}"] = df["volatility"].rolling(w, min_periods=w).mean().astype(np.float32)
        if _need(f"vol_rolling_std_{w}"):
            df[f"vol_rolling_std_{w}"]  = df["volatility"].rolling(w, min_periods=w).std().astype(np.float32)

    for s in EWMA_SPANS:
        if _need(f"vol_ewma_{s}"):
            df[f"vol_ewma_{s}"] = df["volatility"].ewm(span=s, adjust=False).mean().astype(np.float32)

    df = df.iloc[WARMUP_ROWS:].reset_index(drop=True)

    if selected_features:
        cols_to_keep = list(selected_features)
        for raw in ["volatility", "OFI"]:
            if raw not in cols_to_keep:
                cols_to_keep.append(raw)
        df = df[[c for c in cols_to_keep if c in df.columns]]

    return df.astype(np.float32)


def build_targets(df: pd.DataFrame):
    """Build next-step volatility and OFI targets. Returns (X, y_vol, y_ofi, feature_cols)."""
    y_vol = df["volatility"].shift(-1).astype(np.float32)
    y_ofi = df["OFI"].shift(-1).astype(np.float32)
    valid = y_vol.notna()
    df    = df.loc[valid].copy()
    y_vol = y_vol.loc[valid].values
    y_ofi = y_ofi.loc[valid].values
    feature_cols = [c for c in df.columns if c not in ("volatility", "OFI")]
    X = df[feature_cols].values.astype(np.float32)
    return X, y_vol, y_ofi, feature_cols


print(f"{_ts()} ✅ Feature engineering functions defined")
print(f"   LAG_RANGE       : 1..{max(LAG_RANGE)}")
print(f"   ROLLING_WINDOWS : {ROLLING_WINDOWS}")
print(f"   EWMA_SPANS      : {EWMA_SPANS}")
print(f"   WARMUP_ROWS     : {WARMUP_ROWS}")
print(f"   Expected cols   : ~{7 + 2*len(LAG_RANGE) + 2*len(ROLLING_WINDOWS)*2 + 2*len(EWMA_SPANS)} features")

[20:15:49] ✅ Feature engineering functions defined
   LAG_RANGE       : 1..30
   ROLLING_WINDOWS : [5, 10, 15, 20, 30, 50]
   EWMA_SPANS      : [5, 10, 15, 20, 30, 50]
   WARMUP_ROWS     : 50
   Expected cols   : ~103 features


## Cell 3 — Stage 1: Exploratory Feature Selection

In [5]:
# ── Cell 3: Stage 1 — Exploratory Feature Selection (Chunked) ────────────────
# Processes files in chunks to avoid OOM during feature engineering.
# Stage 1 only needs reliable feature importances → aggressive subsampling is fine.

CHUNK_SIZE         = 30   # files per chunk — tune down if still OOM
STAGE1_SUBSAMPLE   = 50   # keep 1 in 50 rows (plenty for importance ranking)

print("=" * 65)
print(f"{_ts()} STAGE 1 — EXPLORATORY FEATURE SELECTION (chunked)")
print("=" * 65)

t_stage1 = time.time()

all_files_list = sorted(glob.glob(DATA_PATH))
n_files  = len(all_files_list)
n_chunks = (n_files + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"\n{_ts()} Files: {n_files} | Chunk size: {CHUNK_SIZE} | Chunks: {n_chunks}")
print(f"   Stage-1 subsample: 1/{STAGE1_SUBSAMPLE} (feature importance only)")

X_list, y_vol_list, y_ofi_list = [], [], []
feature_cols = None

for chunk_idx in range(n_chunks):
    chunk_files = all_files_list[chunk_idx * CHUNK_SIZE : (chunk_idx + 1) * CHUNK_SIZE]
    print(f"\n{_ts()} Chunk {chunk_idx+1}/{n_chunks}  ({len(chunk_files)} files)")

    # Load chunk
    chunk_data = load_and_concat(chunk_files)

    # Aggressive subsample for Stage 1
    chunk_sub = chunk_data[::STAGE1_SUBSAMPLE]
    print(f"   After 1/{STAGE1_SUBSAMPLE} subsample: {chunk_sub.shape[0]:,} rows")
    del chunk_data

    # Build features
    df_chunk = build_features(chunk_sub)
    del chunk_sub

    # Build targets
    X_c, y_vol_c, y_ofi_c, feature_cols = build_targets(df_chunk)
    del df_chunk

    X_list.append(X_c)
    y_vol_list.append(y_vol_c)
    y_ofi_list.append(y_ofi_c)
    del X_c, y_vol_c, y_ofi_c

    import gc; gc.collect()
    total_so_far = sum(len(x) for x in X_list)
    print(f"   {_ts()} Chunk done. Accumulated {total_so_far:,} rows total")

# ── Concatenate ───────────────────────────────────────────────────────────────
X_train     = np.concatenate(X_list,     axis=0)
y_vol_train = np.concatenate(y_vol_list, axis=0)
y_ofi_train = np.concatenate(y_ofi_list, axis=0)
del X_list, y_vol_list, y_ofi_list
gc.collect()

print(f"\n{_ts()} Feature columns : {len(feature_cols)}")
print(f"   Total train rows: {X_train.shape[0]:,}")
print(f"   X_train size    : {X_train.nbytes / 1e9:.2f} GB")

# ── Train exploratory volatility model ───────────────────────────────────────
print(f"\n{_ts()} --- Volatility model (exploratory) ---")
t0 = time.time()
vol_model_exp = lgb.LGBMRegressor(**LGB_STAGE1_PARAMS)
vol_model_exp.fit(
    X_train, y_vol_train,
    callbacks=[lgb.log_evaluation(period=LOG_PERIOD)],
)
print(f"{_ts()} Volatility exploratory done in {time.time()-t0:.1f}s"
      f"  (best iter: {vol_model_exp.best_iteration_})")

# ── Train exploratory OFI model ───────────────────────────────────────────────
print(f"\n{_ts()} --- OFI model (exploratory) ---")
t0 = time.time()
ofi_model_exp = lgb.LGBMRegressor(**LGB_STAGE1_PARAMS)
ofi_model_exp.fit(
    X_train, y_ofi_train,
    callbacks=[lgb.log_evaluation(period=LOG_PERIOD)],
)
print(f"{_ts()} OFI exploratory done in {time.time()-t0:.1f}s"
      f"  (best iter: {ofi_model_exp.best_iteration_})")

del X_train, y_vol_train, y_ofi_train
gc.collect()

# ── Rank by importance ────────────────────────────────────────────────────────
def _top_features(model, cols, label, threshold=CUMULATIVE_THRESHOLD):
    imp        = model.feature_importances_.astype(np.float64)
    total      = imp.sum()
    if total == 0:
        return set(cols)
    order      = np.argsort(imp)[::-1]
    cumulative = 0.0
    selected   = []
    print(f"\n{_ts()} {label} — top features (threshold={threshold*100:.0f}%):")
    for rank, idx in enumerate(order):
        frac        = imp[idx] / total
        cumulative += frac
        selected.append(cols[idx])
        if rank < 10:
            print(f"      {rank+1:3d}. {cols[idx]:35s} {frac*100:5.2f}%  (cum {cumulative*100:5.1f}%)")
        if cumulative >= threshold:
            print(f"      ... {rank+1} features reach {cumulative*100:.1f}% importance")
            break
    return set(selected)

vol_feats = _top_features(vol_model_exp, feature_cols, "Volatility")
ofi_feats = _top_features(ofi_model_exp, feature_cols, "OFI")

selected_features = sorted(vol_feats | ofi_feats)
dropped           = sorted(set(feature_cols) - set(selected_features))

print(f"\n{'='*65}")
print(f"FEATURE SELECTION SUMMARY")
print(f"{'='*65}")
print(f"   Total features   : {len(feature_cols)}")
print(f"   Selected (union) : {len(selected_features)}")
print(f"   Dropped          : {len(dropped)}")
print(f"   Stage 1 total    : {time.time()-t_stage1:.1f}s")


[20:15:49] STAGE 1 — EXPLORATORY FEATURE SELECTION (chunked)

[20:15:49] Files: 230 | Chunk size: 30 | Chunks: 8
   Stage-1 subsample: 1/50 (feature importance only)

[20:15:49] Chunk 1/8  (30 files)
  [20:15:49] [01/30] tensor_OFI_Enhanced_BTC_2023-05-16.pt  shape=(527472, 7)  (0.0s elapsed)
  [20:15:49] [02/30] tensor_OFI_Enhanced_BTC_2023-05-17.pt  shape=(947035, 7)  (0.0s elapsed)
  [20:15:49] [03/30] tensor_OFI_Enhanced_BTC_2023-05-18.pt  shape=(1380768, 7)  (0.0s elapsed)
  [20:15:49] [04/30] tensor_OFI_Enhanced_BTC_2023-05-19.pt  shape=(909060, 7)  (0.0s elapsed)
  [20:15:49] [05/30] tensor_OFI_Enhanced_BTC_2023-05-20.pt  shape=(461962, 7)  (0.0s elapsed)
  [20:15:49] [06/30] tensor_OFI_Enhanced_BTC_2023-05-21.pt  shape=(635300, 7)  (0.0s elapsed)
  [20:15:49] [07/30] tensor_OFI_Enhanced_BTC_2023-05-22.pt  shape=(839344, 7)  (0.0s elapsed)
  [20:15:49] [08/30] tensor_OFI_Enhanced_BTC_2023-05-23.pt  shape=(941171, 7)  (0.0s elapsed)
  [20:15:49] [09/30] tensor_OFI_Enhanced_BTC_20

## Cell 4 — Stage 2: Final Model Training

In [6]:
# ── Cell 4: Stage 2 — Final Models (chunked, memory-safe) ────────────────────
import gc

CHUNK_SIZE      = 10
MAX_STAGE2_ROWS = 2_000_000

# Bump n_estimators so OFI CV can find a true early-stopping point
LGB_PARAMS["n_estimators"] = 6000

print("=" * 65)
print(f"{_ts()} STAGE 2 — FINAL MODELS (chunked, memory-safe)")
print("=" * 65)

t_stage2 = time.time()

all_files_list = sorted(glob.glob(DATA_PATH))
n_files  = len(all_files_list)
n_chunks = (n_files + CHUNK_SIZE - 1) // CHUNK_SIZE
target_per_chunk = max(1, MAX_STAGE2_ROWS // n_chunks)

print(f"\n{_ts()} Files: {n_files} | Chunks: {n_chunks}")
print(f"   Target rows/chunk : ~{target_per_chunk:,}")
print(f"   Hard cap          : {MAX_STAGE2_ROWS:,} rows total")
print(f"   Selected features : {len(selected_features)}")
print(f"   Max estimators    : {LGB_PARAMS['n_estimators']}")

X_list, y_vol_list, y_ofi_list = [], [], []
feature_cols_f       = None
total_rows_collected = 0

for chunk_idx in range(n_chunks):

    if total_rows_collected >= MAX_STAGE2_ROWS:
        print(f"\n{_ts()} Hit MAX_STAGE2_ROWS — stopping at chunk {chunk_idx+1}")
        break

    chunk_files = all_files_list[chunk_idx * CHUNK_SIZE : (chunk_idx + 1) * CHUNK_SIZE]
    print(f"\n{_ts()} Chunk {chunk_idx+1}/{n_chunks}  ({len(chunk_files)} files)")

    chunk_data = load_and_concat(chunk_files)

    step = max(1, len(chunk_data) // target_per_chunk)
    chunk_sub = chunk_data[::step].copy()
    del chunk_data
    gc.collect()
    print(f"   Subsampled 1/{step}: {chunk_sub.shape[0]:,} rows")

    df_chunk = build_features(chunk_sub, selected_features=selected_features)
    del chunk_sub
    gc.collect()

    X_c, y_vol_c, y_ofi_c, feature_cols_f = build_targets(df_chunk)
    del df_chunk
    gc.collect()

    X_list.append(X_c)
    y_vol_list.append(y_vol_c)
    y_ofi_list.append(y_ofi_c)
    total_rows_collected += len(X_c)
    del X_c, y_vol_c, y_ofi_c

    print(f"   Collected so far: {total_rows_collected:,} / {MAX_STAGE2_ROWS:,} rows")

# ── Concatenate ───────────────────────────────────────────────────────────────
X_train_f     = np.concatenate(X_list,     axis=0)
y_vol_train_f = np.concatenate(y_vol_list, axis=0)
y_ofi_train_f = np.concatenate(y_ofi_list, axis=0)
del X_list, y_vol_list, y_ofi_list
gc.collect()

print(f"\n{_ts()} Feature columns : {len(feature_cols_f)}")
print(f"   Total train rows: {X_train_f.shape[0]:,}")
print(f"   X_train size    : {X_train_f.nbytes / 1e9:.3f} GB")

# ── CV to find best iteration ─────────────────────────────────────────────────
def find_best_iter(X, y, label):
    """Run lgb.cv() to find optimal number of boosting rounds."""
    ds = lgb.Dataset(X, label=y)
    print(f"\n{_ts()} Running {N_CV_FOLDS}-fold CV for {label}...")
    t0 = time.time()
    cv_result = lgb.cv(
        LGB_CV_PARAMS,
        ds,
        num_boost_round=LGB_PARAMS["n_estimators"],
        nfold=N_CV_FOLDS,
        stratified=False,
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
            lgb.log_evaluation(period=LOG_PERIOD),
        ],
        return_cvbooster=False,
    )
    mean_key   = [k for k in cv_result.keys() if k.endswith("-mean")][0]
    best_iter  = len(cv_result[mean_key])
    best_score = cv_result[mean_key][-1]
    print(f"{_ts()} {label} CV done in {time.time()-t0:.1f}s")
    print(f"   Best iteration : {best_iter}")
    print(f"   CV RMSE        : {best_score:.6f}")
    return best_iter

best_iter_vol = find_best_iter(X_train_f, y_vol_train_f, "Volatility")
best_iter_ofi = find_best_iter(X_train_f, y_ofi_train_f, "OFI")

# ── Final volatility model ────────────────────────────────────────────────────
print(f"\n{_ts()} --- Final Volatility model (n_estimators={best_iter_vol}) ---")
t0 = time.time()
vol_model_final = lgb.LGBMRegressor(
    **{k: v for k, v in LGB_PARAMS.items() if k != "n_estimators"},
    n_estimators=best_iter_vol,
)
vol_model_final.fit(X_train_f, y_vol_train_f)
print(f"{_ts()} Final volatility done in {time.time()-t0:.1f}s")

# ── Final OFI model ───────────────────────────────────────────────────────────
print(f"\n{_ts()} --- Final OFI model (n_estimators={best_iter_ofi}) ---")
t0 = time.time()
ofi_model_final = lgb.LGBMRegressor(
    **{k: v for k, v in LGB_PARAMS.items() if k != "n_estimators"},
    n_estimators=best_iter_ofi,
)
ofi_model_final.fit(X_train_f, y_ofi_train_f)
print(f"{_ts()} Final OFI done in {time.time()-t0:.1f}s")

print(f"\n{_ts()} Stage 2 complete in {time.time()-t_stage2:.1f}s")

# ── Save ──────────────────────────────────────────────────────────────────────
print(f"\n{_ts()} Saving models...")
vol_model_final.booster_.save_model(str(VOL_MODEL_PATH))
ofi_model_final.booster_.save_model(str(OFI_MODEL_PATH))

with open(FEATURES_PATH, "w") as f:
    json.dump({"selected_features": selected_features}, f, indent=2)

print(f"{_ts()} Saved:")
print(f"   {VOL_MODEL_PATH.resolve()}")
print(f"   {OFI_MODEL_PATH.resolve()}")
print(f"   {FEATURES_PATH.resolve()}  ({len(selected_features)} features)")

del X_train_f, y_vol_train_f, y_ofi_train_f
gc.collect()
print(f"\n{_ts()} Training data freed from memory.")


[20:17:25] STAGE 2 — FINAL MODELS (chunked, memory-safe)

[20:17:25] Files: 230 | Chunks: 23
   Target rows/chunk : ~86,956
   Hard cap          : 2,000,000 rows total
   Selected features : 53
   Max estimators    : 6000

[20:17:25] Chunk 1/23  (10 files)
  [20:17:25] [01/10] tensor_OFI_Enhanced_BTC_2023-05-16.pt  shape=(527472, 7)  (0.0s elapsed)
  [20:17:25] [02/10] tensor_OFI_Enhanced_BTC_2023-05-17.pt  shape=(947035, 7)  (0.0s elapsed)
  [20:17:25] [03/10] tensor_OFI_Enhanced_BTC_2023-05-18.pt  shape=(1380768, 7)  (0.0s elapsed)
  [20:17:25] [04/10] tensor_OFI_Enhanced_BTC_2023-05-19.pt  shape=(909060, 7)  (0.0s elapsed)
  [20:17:25] [05/10] tensor_OFI_Enhanced_BTC_2023-05-20.pt  shape=(461962, 7)  (0.0s elapsed)
  [20:17:25] [06/10] tensor_OFI_Enhanced_BTC_2023-05-21.pt  shape=(635300, 7)  (0.0s elapsed)
  [20:17:25] [07/10] tensor_OFI_Enhanced_BTC_2023-05-22.pt  shape=(839344, 7)  (0.0s elapsed)
  [20:17:25] [08/10] tensor_OFI_Enhanced_BTC_2023-05-23.pt  shape=(941171, 7)  (0.1s

## Cell 4 — Stage 2: Final Model Training & Save

## Cell 6 — `TreeMarketModel` Wrapper (Backtest Interface)

In [7]:
# ── Cell 6: TreeMarketModel — drop-in replacement for LSTMMarketModel ─────────

class TreeMarketModel:
    """
    Drop-in replacement for LSTMMarketModel in backtesting_tree.ipynb.

    Usage:
        model = TreeMarketModel.from_saved(
            vol_path="vol_model_final.txt",
            ofi_path="ofi_model_final.txt",
            features_path="selected_features.json",
        )
        vol_pred, ofi_pred = model.predict_from_window(recent_rows_np)
    """

    def __init__(self, vol_model, ofi_model, selected_features, feature_fn):
        self.vol_model        = vol_model
        self.ofi_model        = ofi_model
        self.selected_features = selected_features
        self.feature_fn       = feature_fn
        self._predict_vol     = vol_model.predict
        self._predict_ofi     = ofi_model.predict

    @classmethod
    def from_saved(
        cls,
        vol_path:      str = "vol_model_final.txt",
        ofi_path:      str = "ofi_model_final.txt",
        features_path: str = "selected_features.json",
    ):
        """Load saved models and feature list from disk."""
        t0 = time.time()
        vol_booster = lgb.Booster(model_file=vol_path)
        ofi_booster = lgb.Booster(model_file=ofi_path)

        with open(features_path) as f:
            meta = json.load(f)
        selected_features = meta["selected_features"]

        elapsed = time.time() - t0
        print(f"{_ts()} ✅ TreeMarketModel loaded ({elapsed:.2f}s)")
        print(f"   Vol model : {vol_path}")
        print(f"   OFI model : {ofi_path}")
        print(f"   Features  : {len(selected_features)} columns")

        return cls(
            vol_model=vol_booster,
            ofi_model=ofi_booster,
            selected_features=selected_features,
            feature_fn=build_features,
        )

    def predict_from_window(self, recent_rows: np.ndarray) -> tuple:
        """
        Predict vol and OFI from a window of recent raw rows.

        Parameters
        ----------
        recent_rows : np.ndarray, shape (W, 7)
            W >= 51  (50 warmup + 1 prediction row)

        Returns
        -------
        (vol_pred, ofi_pred) : (float, float)
        """
        df = self.feature_fn(recent_rows, selected_features=self.selected_features)
        if df.empty:
            return 0.0, 0.0
        feature_cols = [c for c in self.selected_features
                        if c not in ("volatility", "OFI") and c in df.columns]
        last_row = df[feature_cols].iloc[[-1]].values.astype(np.float32)
        return float(self._predict_vol(last_row)[0]), float(self._predict_ofi(last_row)[0])

    def __repr__(self):
        return (f"TreeMarketModel(features={len(self.selected_features)}, "
                f"vol={type(self.vol_model).__name__}, ofi={type(self.ofi_model).__name__})")


print(f"{_ts()} ✅ TreeMarketModel class defined")

[20:57:22] ✅ TreeMarketModel class defined


## Cell 7 — Smoke Test

In [8]:
# ── Cell 7: Smoke Test ──────────────────────────────────────────────────────
# Load the saved models from disk and verify end-to-end prediction works.

print(f"{_ts()} Loading saved models from disk for smoke test...")
tree_model = TreeMarketModel.from_saved(
    vol_path=str(VOL_MODEL_PATH),
    ofi_path=str(OFI_MODEL_PATH),
    features_path=str(FEATURES_PATH),
)
print(f"   {tree_model}")

# Dummy window: 100 rows × 7 columns (random but plausible)
rng         = np.random.default_rng(42)
dummy_window = rng.standard_normal((100, 7)).astype(np.float32)

t0 = time.time()
vol_pred, ofi_pred = tree_model.predict_from_window(dummy_window)
latency_ms = (time.time() - t0) * 1000

print(f"\n{_ts()} Smoke test results:")
print(f"   vol_pred  : {vol_pred:.6f}  (should be > 0 — Softplus equivalent)")
print(f"   ofi_pred  : {ofi_pred:.6f}  (should be in [-1, 1] — Tanh equivalent)")
print(f"   Latency   : {latency_ms:.2f}ms per predict_from_window call")
print(f"\n{_ts()} ✅ All done — models ready for backtesting_tree.ipynb")
print(f"\nUsage in backtesting_tree.ipynb:")
print(f'   model = TreeMarketModel.from_saved(')
print(f'       vol_path="tree/vol_model_final.txt",')
print(f'       ofi_path="tree/ofi_model_final.txt",')
print(f'       features_path="tree/selected_features.json")')
print(f'   vol, ofi = model.predict_from_window(window_array)  # (W, 7) array')

[20:57:22] Loading saved models from disk for smoke test...
[20:57:22] ✅ TreeMarketModel loaded (0.04s)
   Vol model : vol_model_final.txt
   OFI model : ofi_model_final.txt
   Features  : 53 columns
   TreeMarketModel(features=53, vol=Booster, ofi=Booster)

[20:57:22] Smoke test results:
   vol_pred  : 0.000118  (should be > 0 — Softplus equivalent)
   ofi_pred  : -3.046474  (should be in [-1, 1] — Tanh equivalent)
   Latency   : 20.22ms per predict_from_window call

[20:57:22] ✅ All done — models ready for backtesting_tree.ipynb

Usage in backtesting_tree.ipynb:
   model = TreeMarketModel.from_saved(
       vol_path="tree/vol_model_final.txt",
       ofi_path="tree/ofi_model_final.txt",
       features_path="tree/selected_features.json")
   vol, ofi = model.predict_from_window(window_array)  # (W, 7) array
